# Parte 4 — Modelo de Ensamble Avanzado

In [ ]:
# Toca probar q si sirve, debido a que requiere el archivo dataframe_final.pkl, el cual se genera en el notebook parte_1 que si mal no entendí está incompleto.
import warnings
warnings.filterwarnings('ignore')

import pickle
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from matplotlib.backends.backend_pdf import PdfPages
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.metrics import roc_auc_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.metrics import RocCurveDisplay
from sklearn.metrics import PrecisionRecallDisplay
from xgboost import XGBClassifier

with open('dataframe_final.pkl', 'rb') as f:
    df = pickle.load(f)

for col in df.columns:
    if str(df[col].dtype) == 'string':
        df[col] = df[col].astype(str)

objetivo = 'target'

if objetivo not in df.columns:
    objetivo = df.columns[-1]

X = df.drop(columns=[objetivo])
y = df[objetivo]

columnas_numericas = X.select_dtypes(
    include=['int64', 'float64', 'int32', 'float32']
).columns.tolist()

columnas_categoricas = X.select_dtypes(
    include=['object', 'category', 'bool', 'string']
).columns.tolist()

transformador_numerico = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

transformador_categorico = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

preprocesador = ColumnTransformer([
    ('numerico', transformador_numerico, columnas_numericas),
    ('categorico', transformador_categorico, columnas_categoricas)
])

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

modelo_base = XGBClassifier(
    objective='binary:logistic',
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1,
    tree_method='hist'
)

pipeline = Pipeline([
    ('preprocesamiento', preprocesador),
    ('modelo', modelo_base)
])

parametros = {
    'modelo__n_estimators': [200, 300, 400, 500, 700],
    'modelo__max_depth': [3, 4, 5, 6, 7, 8],
    'modelo__learning_rate': [0.01, 0.03, 0.05, 0.07, 0.1],
    'modelo__subsample': [0.6, 0.7, 0.8, 0.9, 1.0],
    'modelo__colsample_bytree': [0.5, 0.6, 0.7, 0.8, 0.9],
    'modelo__gamma': [0, 0.1, 0.2, 0.3, 0.5],
    'modelo__min_child_weight': [1, 2, 3, 5, 7],
    'modelo__reg_alpha': [0, 0.001, 0.01, 0.1, 1],
    'modelo__reg_lambda': [0.5, 1, 1.5, 2, 3]
}

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

busqueda = RandomizedSearchCV(
    estimator=pipeline,
    param_distributions=parametros,
    n_iter=40,
    scoring='f1_weighted',
    cv=cv,
    verbose=2,
    random_state=42,
    n_jobs=-1,
    refit=True
)

busqueda.fit(X_train, y_train)

mejor_modelo = busqueda.best_estimator_

predicciones = mejor_modelo.predict(X_test)

if hasattr(mejor_modelo.named_steps['modelo'], 'predict_proba'):
    probabilidades = mejor_modelo.predict_proba(X_test)[:, 1]
else:
    probabilidades = None

accuracy = accuracy_score(y_test, predicciones)
precision = precision_score(
    y_test,
    predicciones,
    average='weighted'
)

recall = recall_score(
    y_test,
    predicciones,
    average='weighted'
)

f1 = f1_score(
    y_test,
    predicciones,
    average='weighted'
)

if probabilidades is not None:
    roc_auc = roc_auc_score(y_test, probabilidades)
else:
    roc_auc = np.nan

print('\nMejores parámetros:\n')
print(busqueda.best_params_)

print('\nMétricas del modelo:\n')
print(f'Accuracy: {accuracy:.4f}')
print(f'Precision: {precision:.4f}')
print(f'Recall: {recall:.4f}')
print(f'F1 Score: {f1:.4f}')
print(f'ROC AUC: {roc_auc:.4f}')

print('\nReporte de clasificación:\n')
print(classification_report(y_test, predicciones))

joblib.dump(mejor_modelo, 'xgboost_final.pkl')

with PdfPages('reporte_xgboost.pdf') as pdf:

    fig, ax = plt.subplots(figsize=(10, 6))
    ax.axis('off')

    resumen = (
        'MODELO AVANZADO - XGBOOST\n\n'
        f'Accuracy: {accuracy:.4f}\n'
        f'Precision: {precision:.4f}\n'
        f'Recall: {recall:.4f}\n'
        f'F1 Score: {f1:.4f}\n'
        f'ROC AUC: {roc_auc:.4f}\n\n'
        f'Mejores parámetros:\n{busqueda.best_params_}'
    )

    ax.text(
        0.01,
        0.95,
        resumen,
        fontsize=12,
        va='top'
    )

    pdf.savefig(fig, bbox_inches='tight')
    plt.close()

    fig, ax = plt.subplots(figsize=(8, 6))

    matriz = confusion_matrix(
        y_test,
        predicciones
    )

    disp = ConfusionMatrixDisplay(
        confusion_matrix=matriz
    )

    disp.plot(ax=ax)

    ax.set_title('Matriz de Confusión')

    pdf.savefig(fig, bbox_inches='tight')
    plt.close()

    if probabilidades is not None:

        fig, ax = plt.subplots(figsize=(8, 6))

        RocCurveDisplay.from_predictions(
            y_test,
            probabilidades,
            ax=ax
        )

        ax.set_title('Curva ROC')

        pdf.savefig(fig, bbox_inches='tight')
        plt.close()

        fig, ax = plt.subplots(figsize=(8, 6))

        PrecisionRecallDisplay.from_predictions(
            y_test,
            probabilidades,
            ax=ax
        )

        ax.set_title('Curva Precision-Recall')

        pdf.savefig(fig, bbox_inches='tight')
        plt.close()

    modelo_xgb = mejor_modelo.named_steps['modelo']

    importancias = modelo_xgb.feature_importances_

    nombres = mejor_modelo.named_steps[
        'preprocesamiento'
    ].get_feature_names_out()

    ranking = pd.DataFrame({
        'Variable': nombres,
        'Importancia': importancias
    })

    ranking = ranking.sort_values(
        by='Importancia',
        ascending=False
    ).head(20)

    fig, ax = plt.subplots(figsize=(10, 8))

    ax.barh(
        ranking['Variable'][::-1],
        ranking['Importancia'][::-1]
    )

    ax.set_title('Top 20 Variables Más Importantes')
    ax.set_xlabel('Importancia')

    pdf.savefig(fig, bbox_inches='tight')
    plt.close()

print('\nArchivos generados correctamente:')
print('xgboost_final.pkl')
print('reporte_xgboost.pdf')

NotImplementedError: (<StringDtype(storage='python', na_value=nan)>, array(['Alejandra Salazar Galeano', 'Alejandra Salazar Galeano',
       'Alejandra Salazar Galeano', 'Alejandra Salazar Galeano',
       'Alejandra Salazar Galeano', 'Alejandra Salazar Galeano',
       'Alejandra Salazar Galeano', 'Alejandra Salazar Galeano',
       'Alejandra Salazar Galeano', 'Alejandra Salazar Galeano',
       'Alejandra Salazar Galeano', 'Sergio Gomez Osorio',
       'Sergio Gomez Osorio', 'Sergio Gomez Osorio',
       'Sergio Gomez Osorio', 'Sergio Gomez Osorio',
       'Sergio Gomez Osorio', 'Sergio Gomez Osorio',
       'Sergio Gomez Osorio', 'Sergio Gomez Osorio',
       'Sergio Gomez Osorio', 'Sergio Gomez Osorio',
       'Sergio Gomez Osorio', 'Sergio Gomez Osorio', 'Juan David Rubio',
       'Juan David Rubio', 'Juan David Rubio', 'Juan David Rubio',
       'Juan David Rubio', 'Juan David Rubio', 'Juan David Rubio',
       'Juan David Rubio', 'Juan David Rubio',
       'Carlos Andres Diaz M